In [8]:
import imaplib
import email

def fetch_emails(username, password, imap_server):
    mail = imaplib.IMAP4_SSL(imap_server)
    mail.login(username, password)
    mail.select("inbox")
    _, data = mail.search(None, "UNSEEN")
    email_ids = data[0].split()
    emails = []
    for e_id in email_ids:
        _, msg_data = mail.fetch(e_id, "(RFC822)")
        for response_part in msg_data:
            if isinstance(response_part, tuple):
                msg = email.message_from_bytes(response_part[1])
                emails.append(msg)
    return emails

In [9]:
from email.header import decode_header

def extract_email_features(msg):
    subject = decode_header(msg["Subject"])[0][0]
    if isinstance(subject, bytes):
        subject = subject.decode()
    sender = msg["From"]
    body = ""
    links = []
    attachments = []

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            if content_type == "text/plain":
                body += part.get_payload(decode=True).decode()
            elif content_type == "text/html":
                body += part.get_payload(decode=True).decode()
            elif "attachment" in part.get("Content-Disposition", ""):
                attachments.append(part.get_payload(decode=True))
    else:
        body = msg.get_payload(decode=True).decode()

    # Extract links from body
    import re
    links = re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', body)

    return {
        "subject": subject,
        "sender": sender,
        "body": body,
        "links": links,
        "attachments": attachments
    }

In [10]:
from transformers import pipeline

# Load pre-trained BERT model for text classification
phishing_detector = pipeline("text-classification", model="bert-base-uncased")

def detect_phishing_bert(email_body):
    result = phishing_detector(email_body)
    return result[0]["label"] == "PHISHING"  # Assuming "PHISHING" is the label

def detect_phishing_rf_xgboost(features):
    # Train RandomForest/XGBoost on features
    # Example: model.predict([features])
    pass

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


In [11]:
import requests

def analyze_link(url):
    api_key = "05b6b67c751b82bcbe45da840a041290c62710bc532ca8f4ccb142a209ab0ee5"
    url_id = requests.get(f"https://www.virustotal.com/api/v3/urls/{url}", headers={"x-apikey": api_key})
    if url_id.status_code == 200:
        return url_id.json()["data"]["attributes"]["last_analysis_stats"]["malicious"] > 0
    return False

In [12]:
def scan_attachment(attachment):
    api_key = "05b6b67c751b82bcbe45da840a041290c62710bc532ca8f4ccb142a209ab0ee5"
    files = {"file": ("attachment", attachment)}
    response = requests.post("https://www.virustotal.com/api/v3/files", headers={"x-apikey": api_key}, files=files)
    if response.status_code == 200:
        return response.json()["data"]["attributes"]["last_analysis_stats"]["malicious"] > 0
    return False

In [17]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/detect-phishing", methods=["POST"])
def detect_phishing():
    email_data = request.json
    features = extract_email_features(email_data)
    
    # Run phishing detection
    bert_result = detect_phishing_bert(features["body"])
    rf_xgboost_result = detect_phishing_rf_xgboost(features)
    
    # Analyze links
    malicious_links = [link for link in features["links"] if analyze_link(link)]
    
    # Scan attachments
    malicious_attachments = [attachment for attachment in features["attachments"] if scan_attachment(attachment)]
    
    # Combine results
    is_phishing = bert_result or rf_xgboost_result or malicious_links or malicious_attachments
    
    return jsonify({
        "is_phishing": is_phishing,
        "malicious_links": malicious_links,
        "malicious_attachments": malicious_attachments
    })

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

In [13]:
! pip install pyOpenSSL

   ---------------------------------------- 0.0/56.5 kB ? eta -:--:--
   ---------------------------------------- 56.5/56.5 kB 2.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ----- ---------------------------------- 0.4/3.2 MB 8.9 MB/s eta 0:00:01
   ---------- ----------------------------- 0.8/3.2 MB 8.9 MB/s eta 0:00:01
   ------------------ --------------------- 1.5/3.2 MB 10.6 MB/s eta 0:00:01
   ------------------------ --------------- 2.0/3.2 MB 10.5 MB/s eta 0:00:01
   ------------------------ --------------- 2.0/3.2 MB 10.5 MB/s eta 0:00:01
   --------------------------- ------------ 2.2/3.2 MB 7.8 MB/s eta 0:00:01
   ----------------------------------- ---- 2.8/3.2 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------  3.2/3.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 7.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.1.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# app.py - Main FastAPI application
from fastapi import FastAPI, HTTPException, BackgroundTasks, File, UploadFile, Body, Depends
from fastapi.responses import JSONResponse
from pydantic import BaseModel, EmailStr, HttpUrl
import numpy as np
import pandas as pd
import joblib
import uvicorn
import os
import re
import email
import base64
import hashlib
from email.parser import BytesParser
from email.policy import default
from typing import List, Dict, Optional, Any, Union
import requests
from bs4 import BeautifulSoup
import aiohttp
import asyncio
from urllib.parse import urlparse
import whois
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import ssl
import socket
from datetime import datetime, timedelta
import logging
from fastapi.middleware.cors import CORSMiddleware

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("phishing_detector.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("phishing_detector")

# Initialize the FastAPI app
app = FastAPI(
    title="Real-time Email Phishing Detection API",
    description="API for detecting phishing attempts in emails using multiple ML models and security analysis",
    version="1.0.0"
)

# CORS setup
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Update with specific origins in production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Define data models
class EmailContent(BaseModel):
    sender: EmailStr
    subject: str
    body: str
    html_body: Optional[str] = None
    headers: Optional[Dict[str, str]] = None
    links: Optional[List[str]] = None
    attachment_info: Optional[List[Dict[str, str]]] = None
    received_date: Optional[str] = None

class RawEmail(BaseModel):
    raw_content: str

class UrlToCheck(BaseModel):
    url: HttpUrl

class DetectionResult(BaseModel):
    is_phishing: bool
    confidence: float
    model_scores: Dict[str, float]
    risk_factors: List[Dict[str, Any]]
    safe_score: float
    analysis_time: float
    checked_urls: Optional[List[Dict[str, Any]]] = None
    checked_attachments: Optional[List[Dict[str, Any]]] = None

# Global variables for ML models
bert_tokenizer = None
bert_model = None
random_forest_model = None
xgboost_model = None

# Load ML models
def load_models():
    global bert_tokenizer, bert_model, random_forest_model, xgboost_model
    
    # BERT model for text analysis
    model_name = "bert-base-uncased"  # Can be replaced with a fine-tuned phishing model
    bert_tokenizer = BertTokenizer.from_pretrained(model_name)
    bert_model = TFBertForSequenceClassification.from_pretrained(model_name)
    
    # Load other models (in practice these would be trained and saved models)
    # For demo purposes we'll create new instances
    random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
    xgboost_model = xgb.XGBClassifier(objective='binary:logistic', random_state=42)
    
    logger.info("All ML models loaded successfully")

# Email processing functions
def extract_email_data(raw_email_content: str) -> EmailContent:
    """Extract structured data from a raw email"""
    try:
        # Parse the raw email
        if isinstance(raw_email_content, str):
            raw_email_content = raw_email_content.encode()
        
        message = BytesParser(policy=default).parse(raw_email_content)
        
        # Extract basic information
        sender = message.get('From', '')
        subject = message.get('Subject', '')
        date = message.get('Date', '')
        
        # Extract headers
        headers = {k: v for k, v in message.items()}
        
        # Get body content
        body = ""
        html_body = ""
        links = []
        
        if message.is_multipart():
            for part in message.iter_parts():
                content_type = part.get_content_type()
                if content_type == 'text/plain':
                    body += part.get_content()
                elif content_type == 'text/html':
                    html_content = part.get_content()
                    html_body += html_content
                    # Extract links from HTML
                    soup = BeautifulSoup(html_content, 'html.parser')
                    for a_tag in soup.find_all('a'):
                        href = a_tag.get('href')
                        if href and href.startswith(('http://', 'https://')):
                            links.append(href)
        else:
            content_type = message.get_content_type()
            if content_type == 'text/plain':
                body = message.get_content()
            elif content_type == 'text/html':
                html_content = message.get_content()
                html_body = html_content
                # Extract links from HTML
                soup = BeautifulSoup(html_content, 'html.parser')
                for a_tag in soup.find_all('a'):
                    href = a_tag.get('href')
                    if href and href.startswith(('http://', 'https://')):
                        links.append(href)
        
        # Extract information about attachments
        attachment_info = []
        if message.is_multipart():
            for part in message.iter_parts():
                if part.get_content_disposition() == 'attachment':
                    filename = part.get_filename()
                    content_type = part.get_content_type()
                    attachment_info.append({
                        'filename': filename,
                        'content_type': content_type,
                        'size': len(part.get_content())
                    })
        
        return EmailContent(
            sender=sender,
            subject=subject,
            body=body,
            html_body=html_body,
            headers=headers,
            links=links,
            attachment_info=attachment_info,
            received_date=date
        )
    except Exception as e:
        logger.error(f"Error extracting email data: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Failed to parse email: {str(e)}")

# Feature extraction functions
def extract_features_from_email(email_data: EmailContent) -> Dict[str, Any]:
    """Extract features for ML models from email data"""
    features = {}
    
    # Text features
    features['subject_length'] = len(email_data.subject)
    features['body_length'] = len(email_data.body)
    features['subject_has_urgent'] = 1 if any(word in email_data.subject.lower() for word in ['urgent', 'important', 'attention', 'alert']) else 0
    features['subject_has_money'] = 1 if any(word in email_data.subject.lower() for word in ['money', 'cash', 'bank', 'credit', 'debit', 'account']) else 0
    features['body_has_suspicious'] = 1 if any(word in email_data.body.lower() for word in ['password', 'credential', 'verify', 'click', 'link', 'urgent', 'suspicious']) else 0
    
    # Sender features
    sender_domain = email_data.sender.split('@')[-1] if '@' in email_data.sender else ""
    features['sender_domain'] = sender_domain
    features['sender_has_numbers'] = 1 if any(c.isdigit() for c in email_data.sender.split('@')[0]) else 0
    
    # Link features
    features['link_count'] = len(email_data.links) if email_data.links else 0
    features['suspicious_links'] = 0
    if email_data.links:
        for link in email_data.links:
            # Check for URL tricks like misleading domains
            parsed_url = urlparse(link)
            if '@' in parsed_url.netloc or parsed_url.netloc.count('.') > 2:
                features['suspicious_links'] += 1
    
    # Attachment features
    features['has_attachments'] = 1 if email_data.attachment_info and len(email_data.attachment_info) > 0 else 0
    features['attachment_count'] = len(email_data.attachment_info) if email_data.attachment_info else 0
    features['suspicious_attachments'] = 0
    if email_data.attachment_info:
        suspicious_extensions = ['.exe', '.zip', '.bat', '.js', '.vbs', '.jar']
        for attachment in email_data.attachment_info:
            filename = attachment.get('filename', '').lower()
            if any(filename.endswith(ext) for ext in suspicious_extensions):
                features['suspicious_attachments'] += 1
    
    # Header analysis
    features['has_reply_to'] = 1 if email_data.headers and 'Reply-To' in email_data.headers else 0
    if email_data.headers and 'Reply-To' in email_data.headers and '@' in email_data.headers['Reply-To'] and '@' in email_data.sender:
        reply_domain = email_data.headers['Reply-To'].split('@')[-1]
        sender_domain = email_data.sender.split('@')[-1]
        features['different_reply_domain'] = 1 if reply_domain != sender_domain else 0
    else:
        features['different_reply_domain'] = 0
    
    # SPF/DKIM check (simplified)
    features['has_spf'] = 1 if email_data.headers and any('SPF' in k for k in email_data.headers.keys()) else 0
    features['has_dkim'] = 1 if email_data.headers and any('DKIM' in k for k in email_data.headers.keys()) else 0
    
    return features

# URL analysis functions
async def analyze_url(url: str) -> Dict[str, Any]:
    """Analyze a URL for phishing indicators"""
    result = {
        'url': url,
        'is_suspicious': False,
        'risk_score': 0.0,
        'ssl_valid': False,
        'domain_age_days': 0,
        'redirects': False,
        'url_length': len(url),
        'uses_shortener': False,
        'known_shorteners': ['bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 'is.gd'],
        'warnings': []
    }
    
    try:
        # Parse URL
        parsed = urlparse(url)
        domain = parsed.netloc
        
        # Check domain age
        try:
            w = whois.whois(domain)
            if w.creation_date:
                if isinstance(w.creation_date, list):
                    creation_date = w.creation_date[0]
                else:
                    creation_date = w.creation_date
                domain_age = datetime.now() - creation_date
                result['domain_age_days'] = domain_age.days
                if domain_age.days < 30:
                    result['is_suspicious'] = True
                    result['risk_score'] += 0.3
                    result['warnings'].append("Domain registered less than 30 days ago")
        except Exception as e:
            logger.warning(f"Error checking domain age for {domain}: {str(e)}")
            result['warnings'].append("Could not verify domain age")
            result['risk_score'] += 0.1
        
        # Check if URL is a shortener
        ext = tldextract.extract(domain)
        domain_no_sub = f"{ext.domain}.{ext.suffix}"
        if domain_no_sub in result['known_shorteners'] or domain in result['known_shorteners']:
            result['uses_shortener'] = True
            result['is_suspicious'] = True
            result['risk_score'] += 0.25
            result['warnings'].append("Uses a URL shortening service")
        
        # Check SSL certificate
        try:
            hostname = domain
            context = ssl.create_default_context()
            with socket.create_connection((hostname, 443)) as sock:
                with context.wrap_socket(sock, server_hostname=hostname) as ssock:
                    cert = ssock.getpeercert()
                    # Check if cert is valid
                    if cert:
                        result['ssl_valid'] = True
        except Exception as e:
            logger.warning(f"SSL verification failed for {domain}: {str(e)}")
            result['warnings'].append("Invalid or missing SSL certificate")
            result['risk_score'] += 0.2
        
        # Check for redirect
        try:
            async with aiohttp.ClientSession() as session:
                async with session.head(url, allow_redirects=False) as response:
                    if response.status in (301, 302, 303, 307, 308):
                        result['redirects'] = True
                        if 'Location' in response.headers:
                            redirect_url = response.headers['Location']
                            result['redirect_to'] = redirect_url
                            if domain not in redirect_url:
                                result['is_suspicious'] = True
                                result['risk_score'] += 0.15
                                result['warnings'].append("Redirects to different domain")
        except Exception as e:
            logger.warning(f"Error checking redirect for {url}: {str(e)}")
            # Don't penalize if we can't check
        
        # Check for suspicious URL characteristics
        if '@' in url:
            result['is_suspicious'] = True
            result['risk_score'] += 0.4
            result['warnings'].append("URL contains @ symbol")
        
        if parsed.username or parsed.password:
            result['is_suspicious'] = True
            result['risk_score'] += 0.4
            result['warnings'].append("URL contains credentials")
        
        # Excessive subdomains
        subdomain_count = domain.count('.')
        if subdomain_count > 2:
            result['is_suspicious'] = True
            result['risk_score'] += 0.2 * (subdomain_count - 2)
            result['warnings'].append(f"Excessive subdomains ({subdomain_count})")
        
        # Extremely long URL
        if len(url) > 100:
            result['is_suspicious'] = True
            result['risk_score'] += 0.1
            result['warnings'].append("Excessively long URL")
    
    except Exception as e:
        logger.error(f"Error analyzing URL {url}: {str(e)}")
        result['warnings'].append(f"Analysis error: {str(e)}")
        result['is_suspicious'] = True
        result['risk_score'] += 0.1
    
    # Cap risk score at 1.0
    result['risk_score'] = min(result['risk_score'], 1.0)
    
    return result

# Attachment analysis
def analyze_attachment(attachment_info: Dict[str, str]) -> Dict[str, Any]:
    """Analyze attachment for security risks"""
    result = {
        'filename': attachment_info.get('filename', 'unknown'),
        'content_type': attachment_info.get('content_type', 'unknown'),
        'size': attachment_info.get('size', 0),
        'is_suspicious': False,
        'risk_score': 0.0,
        'warnings': []
    }
    
    # Check file extension
    filename = attachment_info.get('filename', '').lower()
    high_risk_extensions = ['.exe', '.bat', '.vbs', '.js', '.jar', '.ps1', '.hta', '.wsf']
    medium_risk_extensions = ['.zip', '.7z', '.rar', '.doc', '.docm', '.xls', '.xlsm', '.ppt', '.pptm']
    
    if any(filename.endswith(ext) for ext in high_risk_extensions):
        result['is_suspicious'] = True
        result['risk_score'] += 0.8
        result['warnings'].append("High-risk file extension")
    
    if any(filename.endswith(ext) for ext in medium_risk_extensions):
        result['is_suspicious'] = True
        result['risk_score'] += 0.5
        result['warnings'].append("Medium-risk file extension that may contain macros or malware")
    
    # Check for double extensions (e.g., invoice.pdf.exe)
    if filename.count('.') > 1:
        parts = filename.split('.')
        if parts[-1] in ['exe', 'bat', 'js', 'vbs']:
            result['is_suspicious'] = True
            result['risk_score'] += 0.7
            result['warnings'].append("Possible double extension to hide executable")
    
    # Check size
    size = attachment_info.get('size', 0)
    if size < 1000 and any(filename.endswith(ext) for ext in high_risk_extensions + medium_risk_extensions):
        result['is_suspicious'] = True
        result['risk_score'] += 0.3
        result['warnings'].append("Suspiciously small file size for type")
    
    # Cap risk score at 1.0
    result['risk_score'] = min(result['risk_score'], 1.0)
    
    return result

# ML prediction functions
def predict_with_bert(email_data: EmailContent) -> float:
    """Use BERT model to predict phishing likelihood based on email text"""
    # In a real implementation, this would use a fine-tuned model
    # For demo purposes, we use a simplified approach
    
    # Combine subject and body
    text = f"{email_data.subject} {email_data.body}"
    
    # Tokenize and prepare for BERT
    inputs = bert_tokenizer(
        text, 
        return_tensors="tf", 
        padding=True, 
        truncation=True, 
        max_length=512
    )
    
    # Make prediction
    outputs = bert_model(inputs)
    predictions = tf.nn.softmax(outputs.logits, axis=-1)
    
    # For demo purposes, we'll assume the second class (index 1) is "phishing"
    # In a real model, this would depend on how the model was trained
    phishing_score = float(predictions[0][1].numpy())
    
    return phishing_score

def predict_with_random_forest(features: Dict[str, Any]) -> float:
    """Use RandomForest to predict phishing likelihood based on extracted features"""
    # In a real implementation, this would use a trained model
    # For demo purposes, we use a simplified approach
    
    # Convert features to a format suitable for the model
    # For demo, we'll manually calculate a score based on risk factors
    
    # These weights would normally come from the trained model
    risk_weights = {
        'subject_has_urgent': 0.2,
        'subject_has_money': 0.15,
        'body_has_suspicious': 0.25,
        'sender_has_numbers': 0.1,
        'suspicious_links': 0.3,
        'suspicious_attachments': 0.4,
        'has_attachments': 0.05,
        'different_reply_domain': 0.3,
        'has_spf': -0.1,  # Negative weight for protective feature
        'has_dkim': -0.1  # Negative weight for protective feature
    }
    
    # Calculate risk score
    risk_score = 0.0
    for feature, weight in risk_weights.items():
        if feature in features:
            if isinstance(features[feature], bool):
                risk_score += weight if features[feature] else 0
            elif isinstance(features[feature], (int, float)):
                if feature in ['suspicious_links', 'suspicious_attachments']:
                    risk_score += min(1.0, features[feature] * weight)
                else:
                    risk_score += weight if features[feature] > 0 else 0
    
    # Normalize to 0-1 range
    return min(1.0, max(0.0, risk_score))

def predict_with_xgboost(features: Dict[str, Any], url_analyses: List[Dict[str, Any]]) -> float:
    """Use XGBoost to predict phishing likelihood based on combined features"""
    # In a real implementation, this would use a trained model
    # For demo purposes, we use a simplified approach
    
    # Combine email features with URL and attachment analysis
    url_risk = sum(url['risk_score'] for url in url_analyses) / max(1, len(url_analyses))
    
    # These weights would normally come from the trained model
    combined_weights = {
        'subject_has_urgent': 0.15,
        'subject_has_money': 0.1,
        'body_has_suspicious': 0.2,
        'suspicious_links': 0.25,
        'link_count': 0.05,
        'suspicious_attachments': 0.3,
        'different_reply_domain': 0.25,
        'url_risk': 0.4
    }
    
    # Add URL risk to features
    combined_features = {**features, 'url_risk': url_risk}
    
    # Calculate risk score
    risk_score = 0.0
    for feature, weight in combined_weights.items():
        if feature in combined_features:
            if isinstance(combined_features[feature], bool):
                risk_score += weight if combined_features[feature] else 0
            elif isinstance(combined_features[feature], (int, float)):
                if feature in ['suspicious_links', 'suspicious_attachments']:
                    risk_score += min(1.0, combined_features[feature] * weight)
                elif feature == 'url_risk':
                    risk_score += combined_features[feature] * weight
                elif feature == 'link_count':
                    risk_score += min(0.2, combined_features[feature] * 0.01) * weight
                else:
                    risk_score += weight if combined_features[feature] > 0 else 0
    
    # Normalize to 0-1 range
    return min(1.0, max(0.0, risk_score))

# API endpoints
@app.on_event("startup")
async def startup_event():
    """Load models on startup"""
    load_models()

@app.post("/analyze/", response_model=DetectionResult)
async def analyze_email(email_data: EmailContent, background_tasks: BackgroundTasks):
    """Analyze an email for phishing attempts"""
    start_time = datetime.now()
    
    try:
        # Extract features
        features = extract_features_from_email(email_data)
        
        # Analyze URLs
        url_analyses = []
        if email_data.links:
            # Process URLs in parallel
            tasks = [analyze_url(url) for url in email_data.links]
            url_analyses = await asyncio.gather(*tasks)
        
        # Analyze attachments
        attachment_analyses = []
        if email_data.attachment_info:
            attachment_analyses = [analyze_attachment(attachment) for attachment in email_data.attachment_info]
        
        # Make predictions with different models
        bert_score = predict_with_bert(email_data)
        rf_score = predict_with_random_forest(features)
        xgb_score = predict_with_xgboost(features, url_analyses)
        
        # Ensemble the predictions (weighted average)
        ensemble_score = (bert_score * 0.4) + (rf_score * 0.3) + (xgb_score * 0.3)
        
        # Determine risk factors
        risk_factors = []
        
        # Text-based risk factors
        if features.get('subject_has_urgent', 0) > 0:
            risk_factors.append({
                'type': 'text',
                'description': 'Subject contains urgent language',
                'severity': 'medium'
            })
        
        if features.get('body_has_suspicious', 0) > 0:
            risk_factors.append({
                'type': 'text',
                'description': 'Body contains suspicious keywords',
                'severity': 'medium'
            })
        
        # Sender risk factors
        if features.get('different_reply_domain', 0) > 0:
            risk_factors.append({
                'type': 'sender',
                'description': 'Reply-To domain different from sender domain',
                'severity': 'high'
            })
        
        # URL risk factors
        for url_analysis in url_analyses:
            if url_analysis['is_suspicious']:
                risk_factors.append({
                    'type': 'url',
                    'description': f"Suspicious URL: {url_analysis['url']}",
                    'details': url_analysis['warnings'],
                    'severity': 'high' if url_analysis['risk_score'] > 0.7 else 'medium'
                })
        
        # Attachment risk factors
        for attachment_analysis in attachment_analyses:
            if attachment_analysis['is_suspicious']:
                risk_factors.append({
                    'type': 'attachment',
                    'description': f"Suspicious attachment: {attachment_analysis['filename']}",
                    'details': attachment_analysis['warnings'],
                    'severity': 'high' if attachment_analysis['risk_score'] > 0.7 else 'medium'
                })
        
        # Calculate analysis time
        analysis_time = (datetime.now() - start_time).total_seconds()
        
        # Prepare final result
        result = DetectionResult(
            is_phishing=ensemble_score > 0.5,
            confidence=max(ensemble_score, 1 - ensemble_score),
            model_scores={
                'bert': bert_score,
                'random_forest': rf_score,
                'xgboost': xgb_score,
                'ensemble': ensemble_score
            },
            risk_factors=risk_factors,
            safe_score=1 - ensemble_score,
            analysis_time=analysis_time,
            checked_urls=url_analyses,
            checked_attachments=attachment_analyses
        )
        
        # Log detection event (in background to not delay response)
        background_tasks.add_task(
            logger.info,
            f"Phishing detection: score={ensemble_score:.2f}, is_phishing={ensemble_score > 0.5}, risks={len(risk_factors)}"
        )
        
        return result
    
    except Exception as e:
        logger.error(f"Error in phishing analysis: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Analysis failed: {str(e)}")

@app.post("/analyze/raw/", response_model=DetectionResult)
async def analyze_raw_email(raw_email: RawEmail, background_tasks: BackgroundTasks):
    """Analyze a raw email for phishing attempts"""
    try:
        # Extract structured data from raw email
        email_data = extract_email_data(raw_email.raw_content)
        
        # Use the standard endpoint for analysis
        return await analyze_email(email_data, background_tasks)
    
    except Exception as e:
        logger.error(f"Error processing raw email: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Raw email processing failed: {str(e)}")

@app.post("/analyze/url/", response_model=Dict[str, Any])
async def analyze_single_url(url_data: UrlToCheck):
    """Analyze a single URL for phishing indicators"""
    try:
        result = await analyze_url(str(url_data.url))
        return result
    
    except Exception as e:
        logger.error(f"Error analyzing URL: {str(e)}")
        raise HTTPException(status_code=500, detail=f"URL analysis failed: {str(e)}")

@app.get("/health/")
async def health_check():
    """Health check endpoint"""
    return {"status": "healthy", "models_loaded": bert_model is not None}

# Run the app
if __name__ == "__main__":
    uvicorn.run("app:app", host="0.0.0.0", port=8000, reload=True)

c:\Users\Tirth\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:149: UserWarning: Field "model_scores" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
INFO:     Will watch for changes in these directories: ['c:\\Users\\Tirth\\Downloads\\Malicious-Web-Content-Detection-Using-Machine-Learning-master\\Extension']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [6212] using WatchFiles
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfiles.main:1 change detected
INFO:watchfile